In [2]:
!echo "Hello World"

In [3]:
!pip -q install ollama datasets tqdm

In [4]:
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [5]:
!OLLAMA_NUM_PARALLEL=2 OLLAMA_MAX_LOADED_MODELS=1 nohup ollama serve >/tmp/ollama.log 2>&1 &
!sleep 3
!curl -s http://127.0.0.1:11434/api/tags

In [6]:
!ollama pull qwen2.5:1.5b

In [7]:
!ollama run qwen2.5:1.5b "Say hi in one line."

In [8]:
from google.colab import drive
import os
drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/sentiment-engine'
DATA_DIR = os.path.join(BASE_DIR, 'datasets')
os.makedirs(DATA_DIR, exist_ok=True)
print(DATA_DIR)

In [9]:
!ls /content/drive/MyDrive/sentiment-engine/

In [10]:
from pathlib import Path
import textwrap

script_path = Path('/content/drive/MyDrive/sentiment-engine/prepare_data_parallel.py')
script_path.parent.mkdir(parents=True, exist_ok=True)

script = textwrap.dedent('''
import json
import re
import os
import time
import ollama
from datasets import load_dataset
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from os import path

try:
    __DIR__ = path.dirname(path.abspath(__file__))
except NameError:
    __DIR__ = os.getcwd()

# --- CONFIGURATION ---
MODEL_NAME = "qwen2.5:1.5b"
OUTPUT_FILENAME = "/content/drive/MyDrive/sentiment-engine/datasets/train_qwen_28_full.jsonl"
CHECKPOINT_META = OUTPUT_FILENAME + ".checkpoint.json"
LIMIT = None  # Set to None for full dataset (~43k rows)
MAX_WORKERS = 2  # Parallel threads for Colab stability
MAX_IN_FLIGHT = MAX_WORKERS * 4
RESUME = True
CHECKPOINT_EVERY = 50  # lower value reduces worst-case loss on disconnect

# --- 1. MAPPINGS ---
id2label = {
    0: "admiration",
    1: "amusement",
    2: "anger",
    3: "annoyance",
    4: "approval",
    5: "caring",
    6: "confusion",
    7: "curiosity",
    8: "desire",
    9: "disappointment",
    10: "disapproval",
    11: "disgust",
    12: "embarrassment",
    13: "excitement",
    14: "fear",
    15: "gratitude",
    16: "grief",
    17: "joy",
    18: "love",
    19: "nervousness",
    20: "optimism",
    21: "pride",
    22: "realization",
    23: "relief",
    24: "remorse",
    25: "sadness",
    26: "surprise",
    27: "neutral",
}

polarity_map = {
    "admiration": "Positive",
    "amusement": "Positive",
    "approval": "Positive",
    "caring": "Positive",
    "desire": "Positive",
    "excitement": "Positive",
    "gratitude": "Positive",
    "joy": "Positive",
    "love": "Positive",
    "optimism": "Positive",
    "pride": "Positive",
    "relief": "Positive",
    "anger": "Negative",
    "annoyance": "Negative",
    "disappointment": "Negative",
    "disapproval": "Negative",
    "disgust": "Negative",
    "embarrassment": "Negative",
    "fear": "Negative",
    "grief": "Negative",
    "nervousness": "Negative",
    "remorse": "Negative",
    "sadness": "Negative",
    "confusion": "Neutral",
    "curiosity": "Neutral",
    "realization": "Neutral",
    "surprise": "Neutral",
    "neutral": "Neutral",
}

def aggregate_polarity(emotions):
    polarities = {polarity_map.get(emotion, "Neutral") for emotion in emotions}
    return polarities.pop() if len(polarities) == 1 else "Mixed"

def _extract_json_object(raw_text):
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass

    fenced_match = re.search(r"```(?:json)?\\s*(\\{.*?\\})\\s*```", raw_text, re.DOTALL)
    if fenced_match:
        try:
            return json.loads(fenced_match.group(1))
        except json.JSONDecodeError:
            pass

    object_match = re.search(r"(\\{.*\\})", raw_text, re.DOTALL)
    if object_match:
        try:
            return json.loads(object_match.group(1))
        except json.JSONDecodeError:
            pass

    return None

def build_default_emotions(label_emotions):
    if not label_emotions:
        return [{"emotion": "neutral", "confidence_score": 1.0}]

    score = round(1.0 / len(label_emotions), 4)
    confs = []
    running_total = 0.0
    for i, emotion in enumerate(label_emotions):
        if i == len(label_emotions) - 1:
            value = round(1.0 - running_total, 4)
        else:
            value = score
            running_total += value
        confs.append({"emotion": emotion, "confidence_score": max(0.0, min(1.0, value))})
    return confs

def normalize_model_emotions(model_data, label_emotions):
    if not isinstance(model_data, dict):
        return build_default_emotions(label_emotions), "Model output parsing failed."

    raw_emotions = model_data.get("emotions", [])
    if not isinstance(raw_emotions, list):
        raw_emotions = []

    label_set = set(label_emotions)
    normalized = []
    used = set()
    for item in raw_emotions:
        if not isinstance(item, dict):
            continue

        raw_name = str(item.get("emotion", "")).strip().lower()
        if raw_name not in label_set or raw_name in used:
            continue

        try:
            raw_score = float(item.get("confidence_score", 0.0))
        except (TypeError, ValueError):
            raw_score = 0.0

        score = max(0.0, min(1.0, raw_score))
        normalized.append({"emotion": raw_name, "confidence_score": score})
        used.add(raw_name)

    if not normalized:
        normalized = build_default_emotions(label_emotions)

    total = sum(item["confidence_score"] for item in normalized)
    if total <= 0:
        normalized = build_default_emotions([item["emotion"] for item in normalized])
    else:
        for item in normalized:
            item["confidence_score"] = round(item["confidence_score"] / total, 4)

    missing = [emotion for emotion in label_emotions if emotion not in used]
    if missing:
        share = round(0.1 / len(missing), 4)
        for emotion in missing:
            normalized.append({"emotion": emotion, "confidence_score": share})

        total = sum(item["confidence_score"] for item in normalized)
        for item in normalized:
            item["confidence_score"] = round(item["confidence_score"] / total, 4)

    reasoning = str(model_data.get("reasoning", "")).strip()
    if not reasoning:
        reasoning = "The text expresses multiple emotions with varying intensity."

    return normalized, reasoning

def load_processed_row_ids(output_file):
    processed = set()
    if not path.exists(output_file):
        return processed

    missing_row_id_count = 0
    with open(output_file, "r") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
                row_id = entry.get("meta", {}).get("row_id")
                if isinstance(row_id, int):
                    processed.add(row_id)
                else:
                    missing_row_id_count += 1
            except json.JSONDecodeError:
                continue

    if missing_row_id_count:
        print(
            f"Warning: Found {missing_row_id_count} existing lines without meta.row_id. "
            "Resume will only skip lines that contain row_id."
        )

    return processed

def write_checkpoint_meta(row_id, completed):
    with open(CHECKPOINT_META, "w") as ck:
        json.dump(
            {
                "ts": time.time(),
                "completed_in_this_run": completed,
                "last_row_id_seen": row_id,
            },
            ck,
        )
        ck.flush()
        os.fsync(ck.fileno())

# --- 3. WORKER FUNCTION ---
def process_row(row_id, row):
    text = row["text"]
    labels = row["labels"]
    label_emotions = [id2label[i] for i in labels]

    polarity = aggregate_polarity(label_emotions)

    prompt = (
        "You are a sentiment labeling assistant. "
        "Use ONLY the provided candidate emotions. "
        "Return STRICT JSON with this schema: "
        '{"emotions":[{"emotion":"<candidate>","confidence_score":0.0}],"reasoning":"..."}. '
        "Rules: confidence_score must be between 0 and 1, include one item per candidate emotion, and scores must sum to 1. "
        f"Candidate emotions: {label_emotions}. "
        f'Text: "{text}"'
    )

    try:
        response = ollama.chat(
            model=MODEL_NAME, messages=[{"role": "user", "content": prompt}]
        )
        raw_content = response["message"]["content"].strip()
        parsed = _extract_json_object(raw_content)
        emotions_conf, reasoning = normalize_model_emotions(parsed, label_emotions)
    except Exception:
        emotions_conf = build_default_emotions(label_emotions)
        reasoning = "The text expresses multiple emotions with varying intensity."

    output_obj = {
        "polarity": polarity,
        "emotions": [
            {
                "emotion": item["emotion"].capitalize(),
                "confidence_score": item["confidence_score"],
            }
            for item in emotions_conf
        ],
        "reasoning": reasoning,
    }

    entry = {
        "instruction": "Analyze the sentiment. Return JSON with polarity, emotions (with confidence_score), and reasoning.",
        "input": text,
        "output": json.dumps(output_obj),
        "meta": {"row_id": row_id},
    }

    return json.dumps(entry)

# --- 4. MAIN EXECUTION ---
if __name__ == "__main__":
    print("Downloading GoEmotions dataset...")
    dataset = load_dataset("go_emotions", split="train")

    if LIMIT:
        dataset = dataset.select(range(LIMIT))
        print(f"Limiting to first {LIMIT} rows for testing.")

    total_rows = len(dataset)
    processed_row_ids = load_processed_row_ids(OUTPUT_FILENAME) if RESUME else set()

    pending_row_ids = [
        row_id for row_id in range(total_rows) if row_id not in processed_row_ids
    ]

    print(
        f"Starting parallel processing with {MAX_WORKERS} workers "
        f"(in-flight: {MAX_IN_FLIGHT})."
    )
    print(
        f"Resume mode: {'ON' if RESUME else 'OFF'} | "
        f"already processed: {len(processed_row_ids)} | pending: {len(pending_row_ids)}"
    )

    if not pending_row_ids:
        print("No pending rows. Dataset output is already complete.")
        raise SystemExit(0)

    write_mode = "a" if RESUME else "w"
    rows_since_flush = 0
    completed = 0

    def submit_next(executor, futures_dict, pending_iter):
        try:
            row_id = next(pending_iter)
        except StopIteration:
            return False

        future = executor.submit(process_row, row_id, dataset[row_id])
        futures_dict[future] = row_id
        return True

    with open(OUTPUT_FILENAME, write_mode) as out_handle, ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:
        pending_iter = iter(pending_row_ids)
        futures = {}

        for _ in range(min(MAX_IN_FLIGHT, len(pending_row_ids))):
            submit_next(executor, futures, pending_iter)

        with tqdm(total=len(pending_row_ids)) as progress_bar:
            while futures:
                done, _ = wait(futures, return_when=FIRST_COMPLETED)
                for future in done:
                    row_id = futures.pop(future)
                    try:
                        line = future.result()
                        out_handle.write(line + "\\n")
                    except Exception as e:
                        print(f"Error processing row_id {row_id}: {e}")

                    completed += 1
                    rows_since_flush += 1
                    progress_bar.update(1)

                    if rows_since_flush >= CHECKPOINT_EVERY:
                        out_handle.flush()
                        os.fsync(out_handle.fileno())
                        write_checkpoint_meta(row_id=row_id, completed=completed)
                        rows_since_flush = 0

                    submit_next(executor, futures, pending_iter)

        out_handle.flush()
        os.fsync(out_handle.fileno())

    write_checkpoint_meta(row_id=pending_row_ids[-1], completed=completed)
    print(f"Saved {completed} new examples to {OUTPUT_FILENAME}.")
    print(f"Checkpoint meta written to {CHECKPOINT_META}")
    print("Done! Ready for Unsloth training.")
''')

script_path.write_text(script, encoding='utf-8')
print(f'Wrote: {script_path}')
print(f'Size: {script_path.stat().st_size} bytes')

In [11]:
!nohup python3 /content/drive/MyDrive/sentiment-engine/prepare_data_parallel.py > /tmp/train.log 2>&1 &

In [47]:
!cat /tmp/ollama.log | tail -20

In [60]:
!cat /tmp/train.log | tail -10

In [14]:
# Use this to grab the first '1400' lines if training force stopped
#!head -n 200 "/content/drive/MyDrive/sentiment-engine/datasets/train_qwen_28_full.jsonl" > "/content/drive/MyDrive/sentiment-engine/datasets/train_qwen_28_full.jsonl.tmp" && mv "/content/drive/MyDrive/sentiment-engine/datasets/train_qwen_28_full.jsonl.tmp" "/content/drive/MyDrive/sentiment-engine/datasets/train_qwen_28_full.jsonl"
# !cat prepare_data_parallel.py
#!pkill -f "prepare_data_parallel.py"
!cat /content/drive/MyDrive/sentiment-engine/datasets/train_qwen_28_full.jsonl | tail -5

In [15]:
!nvidia-smi

In [16]:
# !pkill -f "ollama serve"

In [17]:
# import time, ollama
# from concurrent.futures import ThreadPoolExecutor, as_completed

# def call_once():
#     return ollama.chat(model="qwen2.5:1.5b", messages=[{"role":"user","content":"Short test prompt"}])

# def benchmark(concurrency=4, iterations=20):
#     t0 = time.perf_counter()
#     with ThreadPoolExecutor(max_workers=concurrency) as ex:
#         futures = [ex.submit(call_once) for _ in range(iterations)]
#         # consume results to ensure calls complete
#         for f in as_completed(futures):
#             _ = f.result()
#     t1 = time.perf_counter()
#     elapsed = t1 - t0
#     print(f"concurrency={concurrency}, iterations={iterations}, elapsed={elapsed:.2f}s, TPS={iterations/elapsed:.2f}, avg_latency={(elapsed/iterations):.2f}s")

# # example
# benchmark(concurrency=2, iterations=20)